# Create Fact Table

### Reading Silver Data

In [0]:
df_silver = spark.sql('SELECT * FROM parquet.`abfss://silver@salesprojectmgr.dfs.core.windows.net`')
display(df_silver)

### Reading all Dimension Tables

In [0]:
df_dealer = spark.sql("SELECT * FROM sales_catalog.gold.dim_dealer")
df_branch = spark.sql("SELECT * FROM sales_catalog.gold.dim_branch")
df_model = spark.sql("SELECT * FROM sales_catalog.gold.dim_model")
df_date = spark.sql("SELECT * FROM sales_catalog.gold.dim_date")

### Bringing all the surrogate keys from Dimension Tables to Fact Table

In [0]:
df_fact = df_silver.join(df_branch, df_silver['Branch_ID']==df_branch['Branch_ID'], how='left')\
    .join(df_dealer, df_silver['Dealer_ID']==df_dealer['Dealer_ID'], how='left')\
    .join(df_model, df_silver['Model_ID']==df_model['Model_ID'], how='left')\
    .join(df_date, df_silver['Date_ID']==df_date['Date_ID'], how='left')\
    .select(df_silver['Revenue'], df_silver['Units_Sold'], df_silver['Revenue_per_Unit'], 
            df_branch['dim_branch_key'], df_dealer['dim_dealer_key'], 
            df_model['dim_model_key'], df_date['dim_date_key'])
    
display(df_fact)


### Writting Fact Table

In [0]:
if spark.catalog.tableExists('fact_table'):
    delta_table = DeltaTable.forName(spark, 'sales_catalog.gold.fact_table')

    delta_table.alias('trg').merge(df_fact.alias('src'), 'trg.dim_branch_key = src.dim_branch_key and trg.dim_dealer_key = src.dim_dealer_key and trg.dim_model_key = src.dim_model_key and trg.dim_date_key = src.dim_date_key')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_fact.write.format('delta')\
        .mode('Overwrite')\
        .option("path","abfss://gold@salesprojectmgr.dfs.core.windows.net/fact_table")\
        .saveAsTable('sales_catalog.gold.fact_table')


In [0]:
%sql
SELECT * FROM sales_catalog.gold.fact_table